# Prueba de Concepto (PoC) - Tarea 3: Similitud Semántica y Calificación de Respuestas Cortas en E-Learning con Cross-Encoder

Este notebook contiene la implementación, entrenamiento y evaluación de la prueba de concepto para la **Tarea 3** del proyecto: la evaluación de respuestas de examen cortas escritas por alumnos mediante la medición de la similitud semántica contra una respuesta ideal proporcionada por el docente. Para ello, realizamos el ajuste fino (fine-tuning) de un modelo **Cross-Encoder** basado en la arquitectura Transformer (`cross-encoder/stsb-distilroberta-base`) utilizando el corpus ASAP-SAS.

## Organización del Notebook por Secciones

Para facilitar el seguimiento del flujo de trabajo y la navegación, el notebook se encuentra estructurado en las siguientes secciones lógicas:

| Sección | Celdas Asociadas | Descripción General del Funcionamiento |
| :--- | :---: | :--- |
| **Introducción** | `[0]` | Presentación de la prueba de concepto para la evaluación de respuestas cortas. |
| **Sección 1: Configuración del Entorno** | `[1 - 5]` | Preparación del entorno de ejecución, instalación de dependencias e importación de módulos para modelado de similitud. |
| **Sección 2: Carga y Preparación del Dataset** | `[6 - 7]` | Carga y formateo del corpus ASAP-SAS, normalizando linealmente las valoraciones de los evaluadores humanos. |
| **Sección 3: Generación de Pares para Cross-Encoder** | `[8 - 11]` | Construcción de pares estructurados de entrada asociando cada respuesta con la clave ideal correspondiente por prompt. |
| **Sección 4: Referencia Zero-Shot** | `[12 - 14]` | Evaluación del modelo de similitud base no reentrenado para establecer una línea de rendimiento de referencia. |
| **Sección 5: Fine-Tuning del Cross-Encoder** | `[15 - 19]` | Ajuste de parámetros mediante regresión de similitud semántica, empleando la métrica de correlación de rangos de Spearman. |
| **Sección 6: Evaluación y Validación en Dev** | `[20 - 22]` | Medición formal del rendimiento y la correlación de Spearman lograda tras el alineamiento de dominio. |
| **Sección 7: Auditoría Cualitativa de Rankings** | `[23 - 24]` | Clasificación sistemática de predicciones según su error absoluto para auditar cualitativamente el modelo. |

# Sección 1: Configuración del Entorno y Carga de Dependencias

Esta sección inicial prepara el entorno y carga las librerías necesarias de modelado semántico de Sentence-Transformers y PyTorch. Configura el hardware de GPU disponible y asegura el almacenamiento persistente en Google Drive para resguardar el modelo entrenado y los archivos de métricas, permitiendo la reproducibilidad del experimento en entornos en la nube.

In [1]:
import os
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from sentence_transformers import InputExample
from sentence_transformers.cross_encoder import CrossEncoder
from sentence_transformers.cross_encoder.evaluation import CECorrelationEvaluator
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
import torch
from torch.utils.data import DataLoader

In [20]:
!pip install pandas scikit-learn datasets sentence-transformers transformers torch scipy


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
%cd /content/drive/MyDrive/PLN-PROYECTO/Implementación

/content/drive/MyDrive/PLN-PROYECTO/Implementación


## Sección 2: Carga y Preparación del Dataset (ASAP-SAS)

Esta sección carga el corpus ASAP-SAS y prepara los datos unificando las valoraciones de los dos calificadores humanos para formar un único score de referencia, el cual se normaliza en el rango de 0 a 1. Se realiza una partición de entrenamiento y validación estratificada por pregunta (EssaySet), garantizando la representación equitativa de todos los temas evaluados en ambas particiones. Cabe recalcar que la fuente original de los datos proviene de la competición de Kaggle patrocinada por The Hewlett Foundation (Short Answer Scoring, asap-sas, 2012; accedido en 2026-05-07), la cual originalmente no estaba diseñada para similitud de pares de texto sino para clasificación individual de respuestas cortas.

In [5]:
def prepare_datasets(data_path: str, test_size: float = 0.1) -> DatasetDict:
    """
    Carga y procesa el dataset original para la Tarea 3.
    Suma las puntuaciones, las normaliza al rango [0, 1] y
    convierte el resultado al formato estándar de Hugging Face.
    """
    print(f"Cargando datos desde: {data_path}")
    # El archivo original usa tabulaciones (tsv)
    df = pd.read_csv(data_path, sep='\t', encoding='utf-8')

    # 1. Crear una única puntuación sumando Score1 y Score2
    df['Score1'] = pd.to_numeric(df['Score1'], errors='coerce').fillna(0)
    df['Score2'] = pd.to_numeric(df['Score2'], errors='coerce').fillna(0)
    df['score'] = df['Score1'] + df['Score2']

    # 2. Normalizar la puntuación al rango [0, 1]
    # (La nota máxima teórica según Score1+Score2 es 6)
    df['normalized_score'] = df['score'] / 6.0

    # Seleccionamos las columnas de interés:
    # Id de respuesta, Pregunta (EssaySet), Texto de la respuesta y el Score normalizado
    df_clean = df[['Id', 'EssaySet', 'EssayText', 'normalized_score']].copy()

    # 3. Separar un porcentaje para test/dev (10% por defecto)
    # Usamos stratify por 'EssaySet' para asegurar que ambos conjuntos
    # tengan representación proporcional de todas las preguntas.
    train_df, dev_df = train_test_split(
        df_clean,
        test_size=test_size,
        random_state=42,
        stratify=df_clean['EssaySet']
    )

    # 4. Convertir a formato Dataset de HuggingFace
    # (El formato más recomendado para compatibilidad con SBERT / Transformers)
    train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
    dev_dataset = Dataset.from_pandas(dev_df, preserve_index=False)

    hf_datasets = DatasetDict({
        'train': train_dataset,
        'dev': dev_dataset
    })

    print("\n--- Resumen del Dataset ---")
    print(hf_datasets)

    return hf_datasets


## Sección 3: Generación de Pares para el Cross-Encoder

Esta sección implementa el emparejamiento de respuestas para la arquitectura Cross-Encoder. Dado que el corpus original no contenía pares de textos, se extrae la respuesta con la puntuación máxima para cada tipo de pregunta del conjunto de entrenamiento y se define como la solución de referencia (solución ideal), emparejando cada comentario del estudiante con dicha clave para que el modelo procese el alineamiento semántico conjunto mediante atención cruzada.

In [6]:
def create_cross_encoder_pairs(hf_datasets: DatasetDict):
    """
    Toma el dataset preparado y genera los pares (Texto A, Texto B)
    para entrenar el Cross-Encoder.
    Texto A = Respuesta ideal (la de mayor puntuación en el set de entrenamiento para esa pregunta).
    Texto B = Respuesta del alumno a evaluar.
    """
    print("\nGenerando pares para el Cross-Encoder...")
    # Convertimos a pandas para operar fácilmente y buscar el máximo
    train_df = hf_datasets['train'].to_pandas()

    # Encontrar el índice de la respuesta con mayor 'normalized_score' para cada 'EssaySet'
    idx_max_scores = train_df.groupby('EssaySet')['normalized_score'].idxmax()
    best_answers = train_df.loc[idx_max_scores]

    # Diccionario para mapear rápido: EssaySet -> Texto de la mejor respuesta
    ideal_answers_dict = dict(zip(best_answers['EssaySet'], best_answers['EssayText']))

    train_examples = []
    for row in hf_datasets['train']:
        essay_set = row['EssaySet']
        text_a = ideal_answers_dict[essay_set]
        text_b = row['EssayText']
        score = row['normalized_score']

        # SBERT requiere objetos InputExample
        train_examples.append(InputExample(texts=[text_a, text_b], label=float(score)))

    dev_examples = []
    for row in hf_datasets['dev']:
        essay_set = row['EssaySet']
        # Usamos el ideal_answers_dict construido desde 'train' para evitar filtración de datos de 'dev'
        text_a = ideal_answers_dict[essay_set]
        text_b = row['EssayText']
        score = row['normalized_score']

        dev_examples.append(InputExample(texts=[text_a, text_b], label=float(score)))

    print(f"Generados {len(train_examples)} pares de entrenamiento.")
    print(f"Generados {len(dev_examples)} pares de validación.")

    return train_examples, dev_examples, ideal_answers_dict


In [7]:
base_dir = os.getcwd()
train_path = os.path.join(base_dir, "T3", "data", "train.tsv", "train.tsv")


In [8]:
if not os.path.exists(train_path):
  print(f"Error: No se encontró el archivo de datos en {train_path}")
else:
  # Generar el dataset de Hugging Face
  hf_data = prepare_datasets(train_path)

  # Opcional: Guardarlo a disco para que el paso posterior de emparejamiento
  # (Texto A + Texto B) solo tenga que leer este dataset ya limpio y estructurado.
  processed_dir = os.path.join(base_dir, "data", "hf_dataset_clean")
  hf_data.save_to_disk(processed_dir)
  print(f"\nDataset limpio guardado exitosamente en: {processed_dir}")

  # --- NUEVO PASO: Generación de Pares ---
  train_examples, dev_examples, ideal_answers = create_cross_encoder_pairs(hf_data)

  # SBERT entrena usualmente iterando a través de un DataLoader
  train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)

  print("\n¡Preparación de datos completada!")


Cargando datos desde: /content/drive/MyDrive/PLN-PROYECTO/Implementación/T3/data/train.tsv/train.tsv

--- Resumen del Dataset ---
DatasetDict({
    train: Dataset({
        features: ['Id', 'EssaySet', 'EssayText', 'normalized_score'],
        num_rows: 15486
    })
    dev: Dataset({
        features: ['Id', 'EssaySet', 'EssayText', 'normalized_score'],
        num_rows: 1721
    })
})


Saving the dataset (0/1 shards):   0%|          | 0/15486 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1721 [00:00<?, ? examples/s]


Dataset limpio guardado exitosamente en: /content/drive/MyDrive/PLN-PROYECTO/Implementación/data/hf_dataset_clean

Generando pares para el Cross-Encoder...
Generados 15486 pares de entrenamiento.
Generados 1721 pares de validación.

¡Preparación de datos completada!


## Sección 4: Referencia Zero-Shot con Cross-Encoder Preentrenado

Esta sección evalúa el rendimiento del modelo base 'stsb-distilroberta-base' de forma directa sin entrenamiento de dominio. Esta línea base (zero-shot) establece la correlación inicial mediante la métrica del evaluador CECorrelationEvaluator, permitiendo contrastar cuantitativamente el impacto y beneficio del subsiguiente ajuste fino en el dominio educativo.

# Referencia zero-shot

In [9]:
# --- EVALUACIÓN (SIN ENTRENAMIENTO) ---
print("\n--- Iniciando Evaluación Zero-Shot ---")

# 1. Cargar un modelo preentrenado directamente
# 'cross-encoder/stsb-distilroberta-base' está preentrenado para similitud semántica (rango 0 a 1)
model_name = 'cross-encoder/stsb-distilroberta-base'
print(f"Cargando modelo Cross-Encoder preentrenado: {model_name}...")

# Omitimos el warning de pesos si usamos un modelo ya preparado para esta tarea
model = CrossEncoder(model_name, num_labels=1)

# 2. Extraer textos y etiquetas de los InputExamples de validación (dev)
# El evaluador necesita una lista de pares de listas y una lista de notas.
dev_sentence_pairs = [[ex.texts[0], ex.texts[1]] for ex in dev_examples]
dev_scores = [ex.label for ex in dev_examples]

# 3. Configurar el Evaluador
evaluator = CECorrelationEvaluator(
    dev_sentence_pairs,
    dev_scores,
    name='dev-zero-shot'
)

# 4. Ejecutar la evaluación pasándole el modelo
print("\nCalculando predicciones y métricas de correlación...")
# El evaluador calcula automáticamente las predicciones y devuelve las métricas
# Además, guardará los resultados detallados en una carpeta llamada 'eval'
metrics = evaluator(model, output_path=os.path.join(base_dir, "eval"))

print("\n" + "="*50)
# Extraemos spearman si es un dict, sino lo tratamos como float
if isinstance(metrics, dict):
    spearman_score = metrics.get('spearman', metrics.get('dev-zero-shot_spearman', list(metrics.values())[0] if metrics else 0.0))
    print(f"Métricas devueltas: {metrics}")
else:
    spearman_score = metrics

print(f"Resultado de la Correlación de Spearman: {float(spearman_score):.4f}")
print("="*50)

# 5. Muestra de predicciones manuales para ver los datos crudos
print("\n--- Muestra de las primeras 3 predicciones ---")
sample_pairs = dev_sentence_pairs[:3]
sample_labels = dev_scores[:3]
sample_preds = model.predict(sample_pairs)

for i in range(3):
    print(f"Par {i+1}:")
    print(f"  Nota Real:     {sample_labels[i]:.4f}")
    print(f"  Predicción:    {sample_preds[i]:.4f}")
    print(f"  Respuesta:     {sample_pairs[i][1][:60]}...")
    print("-" * 30)



--- Iniciando Evaluación Zero-Shot ---
Cargando modelo Cross-Encoder preentrenado: cross-encoder/stsb-distilroberta-base...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/607 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/328M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.56M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.01k [00:00<?, ?B/s]

/tmp/ipykernel_1978/2164906310.py:18: DeprecationWarning: The CECorrelationEvaluator has been renamed to CrossEncoderCorrelationEvaluator. Please use CrossEncoderCorrelationEvaluator instead.
  evaluator = CECorrelationEvaluator(



Calculando predicciones y métricas de correlación...

Métricas devueltas: {'dev-zero-shot_pearson': 0.30118526865203127, 'dev-zero-shot_spearman': 0.28072598976258534}
Resultado de la Correlación de Spearman: 0.2807

--- Muestra de las primeras 3 predicciones ---
Par 1:
  Nota Real:     0.0000
  Predicción:    0.4151
  Respuesta:     In diffusion the surface of the cell decised what substances...
------------------------------
Par 2:
  Nota Real:     0.0000
  Predicción:    0.6570
  Respuesta:     Protein synthesis has four major steps which are the mRNA le...
------------------------------
Par 3:
  Nota Real:     0.3333
  Predicción:    0.4489
  Respuesta:     I think it is the ablilty to care for others,because Rose ha...
------------------------------


## Sección 5: Fine-Tuning del Cross-Encoder con Trainer API

Esta sección configura y ejecuta el ajuste fino del Cross-Encoder planteando la tarea matemáticamente como una regresión continua con una sola variable de salida. El modelo no se diseña para automatizar calificaciones absolutas por razones de ética, sesgo y explicabilidad, sino para rankear (ordenar) las respuestas del alumnado respecto a la solución ideal y priorizar la supervisión docente. El rendimiento se monitoriza utilizando el coeficiente de correlación de Spearman, una métrica que evalúa la concordancia monótona de los rangos relativos, asegurando una validación justa del orden predicho.

# Fine-Tuning

In [ ]:
print("\n--- INICIANDO ENTRENAMIENTO MODERNO (Trainer API) ---")

model_name = 'cross-encoder/stsb-distilroberta-base'

# 1. Cargar Tokenizador y Modelo Base
# num_labels=1 indica explícitamente que es una tarea de regresión (salida continua)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1)

        # 2. Función de Preprocesamiento (Tokenización de pares)
def preprocess_function(examples):
    """
    Toma un lote (batch) del dataset, busca la respuesta ideal correspondiente
    a cada EssaySet, y tokeniza el par (Texto Ideal, Texto Alumno) junto.
    """
    ideal_texts = [ideal_answers[essay_set] for essay_set in examples['EssaySet']]
    student_texts = examples['EssayText']

    # Tokenizamos ambos textos juntos. El modelo insertará el token separador (e.g., <s> Ideal </s></s> Alumno </s>)
    tokenized = tokenizer(
                ideal_texts,
                student_texts,
                padding="max_length",
                truncation=True,
                max_length=512 # Longitud máxima estándar para RoBERTa
            )

            # El Trainer de HF busca específicamente una columna llamada 'labels'
    tokenized['labels'] = examples['normalized_score']
    return tokenized

        # Aplicamos la tokenización a todo el dataset de forma eficiente (por lotes)
print("Tokenizando los datasets...")
tokenized_datasets = hf_data.map(preprocess_function, batched=True)

# Eliminamos columnas de texto crudo que el modelo matemático no puede procesar
# (Nota: 'score' no está incluido en hf_data porque lo filtramos en df_clean)
tokenized_datasets = tokenized_datasets.remove_columns(['Id', 'EssaySet', 'EssayText', 'normalized_score'])
tokenized_datasets.set_format("torch")

In [15]:
def compute_metrics(eval_pred):
  """Calcula la correlación de Spearman durante la evaluación."""
  predictions, labels = eval_pred
  # Las predicciones vienen con forma (batch_size, 1), las aplanamos a (batch_size,)
  predictions = predictions.squeeze()

  # spearmanr devuelve un objeto, nos quedamos con el atributo 'statistic'
  spearman = spearmanr(labels, predictions).statistic
  return {"spearman": spearman}



In [ ]:
training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",             # Evaluar al final de cada epoch (eval_strategy en lugar de evaluation_strategy en versiones nuevas)
            save_strategy="epoch",             # Guardar un checkpoint al final de cada epoch
            learning_rate=2e-5,                # Tasa de aprendizaje bajita para fine-tuning
            per_device_train_batch_size=16,    # Tamaño del batch (reduce a 8 si te quedas sin memoria)
            per_device_eval_batch_size=16,
            num_train_epochs=4,                # Número de pasadas por todo el dataset
            weight_decay=0.01,                 # Regularización para evitar sobreajuste
            load_best_model_at_end=True,       # Al terminar, quedarse con el modelo que tuvo mejor métrica
            metric_for_best_model="spearman",  # Métrica guía para elegir el mejor modelo
            greater_is_better=True,            # En Spearman, más cerca de 1.0 es mejor
            fp16=torch.cuda.is_available(),    # Precisión mixta solo si hay GPU disponible para evitar cuelgues en CPU
            logging_steps=10                   # Imprimir pérdida cada 10 pasos
        )

        # 5. Instanciar el Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["dev"],
    compute_metrics=compute_metrics,
    # Early stopping: detiene el entrenamiento si pasan 2 epochs sin mejorar el Spearman
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# 6. ¡Entrenar!
print("Iniciando el entrenamiento...")
trainer.train()

# 7. Guardar el modelo final
final_model_path = os.path.join(base_dir, "models", "final_essay_scorer")
trainer.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)
print(f"Entrenamiento completado. Modelo guardado en {final_model_path}")


## Sección 6: Evaluación y Validación Final en Dev

Esta sección valida formalmente el modelo una vez ajustado, calculando la correlación de Spearman final en el conjunto de desarrollo. Esta métrica de correlación permite corroborar cuantitativamente la alineación al dominio lograda por el fine-tuning, superando significativamente el comportamiento del modelo de similitud genérica.

## Evaluación de la tarea

In [22]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments
import torchvision.io
try:
    from torchvision.io import VideoReader
except ImportError:
    class MockVideoReader: pass
    torchvision.io.VideoReader = MockVideoReader

# 1. Rutas y Carga del modelo final
final_model_path = os.path.join(base_dir, "models", "final_essay_scorer")
print(f"Cargando modelo desde: {final_model_path}")

loaded_model = AutoModelForSequenceClassification.from_pretrained(final_model_path)
loaded_tokenizer = AutoTokenizer.from_pretrained(final_model_path)

# 2. Tokenización del conjunto de validación
def preprocess_eval(examples):
    # ideal_answers debe estar definida globalmente
    ideal_texts = [ideal_answers[essay_set] for essay_set in examples['EssaySet']]
    student_texts = examples['EssayText']
    return loaded_tokenizer(ideal_texts, student_texts, padding="max_length", truncation=True, max_length=512)

print("Preparando el dataset de evaluación...")
eval_dataset = hf_data['dev'].map(preprocess_eval, batched=True)
# Aseguramos que la columna labels esté presente y formateada correctamente
if 'labels' not in eval_dataset.column_names:
    eval_dataset = eval_dataset.add_column("labels", hf_data['dev']['normalized_score'])

eval_dataset.set_format("torch", columns=['input_ids', 'attention_mask', 'labels'])

# 3. Configuración de evaluación y Trainer
eval_args = TrainingArguments(
    output_dir="./temp_eval",
    per_device_eval_batch_size=16,
    fp16=torch.cuda.is_available(),
    report_to="none" # Evitar errores de logging externos
)

# Usamos compute_metrics definida anteriormente
trainer_eval = Trainer(
    model=loaded_model,
    args=eval_args,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics
)

print("Ejecutando evaluación final...")
try:
    eval_results = trainer_eval.evaluate()
    print("\n" + "="*50)
    print(f"Resultado Spearman en Validación: {eval_results.get('eval_spearman', 0):.4f}")
    print("="*50)
except Exception as e:
    print(f"Error durante la evaluación: {e}")

Cargando modelo desde: /content/drive/MyDrive/PLN-PROYECTO/Implementación/models/final_essay_scorer


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Preparando el dataset de evaluación...


Map:   0%|          | 0/1721 [00:00<?, ? examples/s]

Ejecutando evaluación final...



Resultado Spearman en Validación: 0.8378


## Sección 7: Auditoría Cualitativa y Análisis de Rankings

Esta sección audita cualitativamente las predicciones ordenándolas por su error absoluto en validación. Agrupa los ejemplos en predicciones con error mínimo (ranking casi perfecto), mediano y máximo para analizar los límites semánticos del modelo frente a respuestas extremadamente cortas, uso de sinónimos técnicos infrecuentes o errores de sintaxis.

In [23]:
import pandas as pd

# 1. Obtener predicciones para todo el conjunto de validación
print("Calculando predicciones detalladas para la auditoría...")
predictions_output = trainer_eval.predict(eval_dataset)
preds = predictions_output.predictions.squeeze()
labels = predictions_output.label_ids

# 2. Crear un DataFrame para facilitar el análisis
audit_df = hf_data['dev'].to_pandas()
audit_df['prediction'] = preds
audit_df['abs_error'] = abs(audit_df['normalized_score'] - audit_df['prediction'])

# 3. Clasificar ejemplos
# Ordenar por error para ver los mejores y peores
audit_df = audit_df.sort_values(by='abs_error')

def print_audit_examples(df, title, num_examples=3):
    print(f"\n{'='*20} {title} {'='*20}")
    for i, row in df.head(num_examples).iterrows():
        print(f"ID: {row['Id']} | Pregunta (Set): {row['EssaySet']}")
        print(f"Nota Real: {row['normalized_score']:.4f} | Predicción: {row['prediction']:.4f} | Error: {row['abs_error']:.4f}")
        print(f"Texto: {row['EssayText'][:200]}...")
        print("-"*60)

# Mejores (Error mínimo)
print_audit_examples(audit_df, "EJEMPLOS: RANKING CASI PERFECTO")

# Regulares (Error en el percentil 50)
mid_point = len(audit_df) // 2
print_audit_examples(audit_df.iloc[mid_point:], "EJEMPLOS: RANKING REGULAR", 3)

# Peores (Error máximo)
print_audit_examples(audit_df.sort_values(by='abs_error', ascending=False), "EJEMPLOS: MÁS DIFÍCILES DE RANKEAR")

Calculando predicciones detalladas para la auditoría...



==================== EJEMPLOS: RANKING CASI PERFECTO ====================
ID: 11502 | Pregunta (Set): 5
Nota Real: 0.0000 | Predicción: 0.0000 | Error: 0.0000
Texto: Anaphase, Interphase,Metaphase,Prophase...
------------------------------------------------------------
ID: 14789 | Pregunta (Set): 6
Nota Real: 0.0000 | Predicción: -0.0001 | Error: 0.0001
Texto: First would be that the cell membrane would squish in different direction to allow substances to cause no harm. Second would be that maybe the membrane would do the exact opisite and would resist any ...
------------------------------------------------------------
ID: 12071 | Pregunta (Set): 5
Nota Real: 0.0000 | Predicción: 0.0001 | Error: 0.0001
Texto: slides over nucleus makesa new cell...
------------------------------------------------------------

==================== EJEMPLOS: RANKING REGULAR ====================
ID: 23945 | Pregunta (Set): 9
Nota Real: 0.0000 | Predicción: 0.0736 | Error: 0.0736
Texto: This chapter was t